![display relevant image here](path/url/to/image)
- Banner/header image

# Title
- Relevant to Data and Business Context

## Overview
- BLUF (Bottom Line Up Front)
- One paragraph summary of findings and analysis
- Frame your 'story'

## Business Understanding
- Set the stage for analysis
- Why are these findings relevant/important?
- Introduce stakeholders
- Postulate about questions you want to ask/answer

## Data Understanding
- Present the source of data
- Describe the data available
- What is relevant to keep what is not
- Present any data cleaning that needs to happen
- Null values? Type mismatches? Duplicates?

***Business Understanding***

The rapid growth of the mobile application industry has created significant opportunities for startups seeking to enter the digital marketplace. This project focuses on analyzing trends within the Google Play Store to help a startup company better understand the characteristics associated with successful Android applications.

The primary stakeholders for this analysis include:

Startup founders and business managers
Product development teams
Marketing teams
Investors and strategic planners

The goal of this analysis is to identify trends related to:

App popularity
User ratings
Install counts
Pricing strategies
Category performance
Monetization opportunities

Understanding these trends can help the company make data-driven decisions regarding:

Which app categories to target
Whether free or paid models perform better
What features users value most
How competition differs across categories

Several analytical questions were formulated to guide the project:

Which app categories receive the highest number of installs?
What relationship exists between app ratings and install counts?
Are free apps more successful than paid apps?
Which categories generate the highest user engagement?
What pricing trends exist among successful paid apps?
Which app categories have the highest average ratings?
How does app size relate to user ratings and installs?

These questions will guide the exploratory analysis, data cleaning, and dashboard development throughout the CRISP-DM process.

## Importing Required Libraries

This section imports all libraries required for the project.

- PySpark is used for large-scale data processing and analysis.
- Pandas is included for possible lightweight data manipulation tasks.
- Matplotlib is imported for optional notebook visualizations during exploratory analysis.

In [1]:
# Imports
# PySpark Session
from pyspark.sql import SparkSession

# PySpark Functions
from pyspark.sql.functions import *

# PySpark Data Types
from pyspark.sql.types import *

# Data Manipulation
import pandas as pd

# Visualization 
import matplotlib.pyplot as plt

## Creating Spark Session and Loading Data

A Spark Session is created to initialize the PySpark environment and allow distributed data processing.

The Google Play Store dataset is then loaded into a PySpark DataFrame.

Key settings used:
- `header=True` ensures the first row is treated as column names.
- `inferSchema=True` allows PySpark to automatically detect data types.

The dataset is now ready for exploratory data analysis (EDA) and cleaning.

In [3]:
# EDA Code Here - Create New Cells As Needed
# Create Spark Session
spark = SparkSession.builder \
    .appName("GooglePlayStoreAnalysis") \
    .master("local[*]") \
    .getOrCreate()
    
    # Load dataset
df = spark.read.csv(
    "google_play_store_dataset.csv",
    header=True,
    inferSchema=True
)

## Data Preparation



## Checking for Missing Values

Before cleaning the dataset, it is important to identify missing or null values.

This step helps determine:
- Which columns contain incomplete data
- The extent of missing information
- What cleaning actions may be required later

Understanding null values is essential for maintaining data quality and ensuring accurate analysis.

In [4]:
# Data Prep Code Here
# Check null values in each column
from pyspark.sql.functions import col, sum, when

null_counts = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

null_counts.show()

+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+
|App|Category|Rating|Reviews|Size|Installs|Type|Price|Content Rating|Genres|Last Updated|Current Ver|Android Ver|
+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+
|  0|       0|     0|      0|   0|       0|   0|    0|             1|     0|           0|          1|          1|
+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+



## Checking for Duplicate Records

Duplicate records can negatively affect analysis results by inflating counts and creating misleading insights.

This step checks whether the dataset contains repeated rows.

Identifying duplicates helps improve:
- Data accuracy
- Reliability of analysis
- Overall data quality

In [5]:
# Check duplicate rows
duplicate_count = df.count() - df.dropDuplicates().count()

print("Number of Duplicate Rows:", duplicate_count)

Number of Duplicate Rows: 483


## Removing Duplicate Records

After identifying duplicate rows, the duplicates are removed from the dataset.

Removing duplicates helps ensure:
- More accurate calculations
- Reliable visualizations
- Better overall data consistency

The updated row count is then displayed to confirm the cleaning process.

In [6]:
# Remove duplicate rows
df = df.dropDuplicates()

print("Rows after removing duplicates:", df.count())

Rows after removing duplicates: 10358


## Reviewing Dataset Schema

The dataset schema is displayed to examine the structure of the DataFrame after initial cleaning.

This step helps verify:
- Column names
- Data types
- Whether columns were loaded correctly

Reviewing the schema is important before performing further data cleaning and transformations.

In [7]:
# Print schema again
df.printSchema()

root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



## Cleaning the Installs Column

The `Installs` column contains special characters such as `+` signs and commas, which prevent proper numerical analysis.

In this step:
- Unnecessary characters are removed
- The column is converted into an integer data type

This allows the column to be used for calculations, filtering, and visualizations.

In [8]:
# Clean Installs column
df = df.withColumn(
    "Installs",
    regexp_replace(col("Installs"), "[+,]", "")
)

# Convert to integer
df = df.withColumn(
    "Installs",
    col("Installs").cast("int")
)

## Cleaning the Price Column

The `Price` column contains dollar signs (`$`) that prevent the values from being treated as numeric data.

In this step:
- Dollar signs are removed
- The column is converted into a float data type

This allows price values to be used for calculations, comparisons, and visual analysis.

In [9]:
# Clean Price column
df = df.withColumn(
    "Price",
    regexp_replace(col("Price"), "[$]", "")
)

# Convert to float
df = df.withColumn(
    "Price",
    col("Price").cast("float")
)

## Verifying Cleaned Columns

After cleaning the `Installs` and `Price` columns, the next step is to verify that the transformations were successful.

This check helps confirm:
- Special characters were removed correctly
- Data type conversions worked properly
- The columns are ready for analysis and visualization

In [10]:
# Verify cleaned columns
df.select("Installs", "Price").show(10)

+---------+-----+
| Installs|Price|
+---------+-----+
|   100000|  0.0|
|100000000|  0.0|
|  5000000|  0.0|
|  5000000|  0.0|
|   100000|  0.0|
|   100000|  0.0|
| 10000000|  0.0|
| 10000000|  0.0|
| 50000000|  0.0|
| 50000000|  0.0|
+---------+-----+
only showing top 10 rows


## Checking Unique Rating Values

This step examines all distinct values in the `Rating` column.

It helps to:
- Understand how ratings are distributed
- Detect unexpected or invalid values (e.g., symbols, text, or out-of-range numbers)
- Verify that previous cleaning steps were successful

This is part of validating data quality before analysis and visualization.

In [12]:
# View distinct Rating values
df.select("Rating").distinct().show(20)

+------+
|Rating|
+------+
|   1.0|
|   2.6|
|   3.1|
|   4.2|
|   4.4|
|   3.8|
|   2.7|
|   1.7|
|   2.9|
|   4.5|
|   2.5|
|   2.4|
|   4.9|
|   3.4|
|   1.6|
|   3.3|
|   1.8|
|   4.3|
|   3.5|
|   4.8|
+------+
only showing top 20 rows


## Converting Rating Column to Float

The `Rating` column is converted into a numeric (float) data type.

This ensures:
- Ratings can be used in calculations (mean, max, min, etc.)
- Proper statistical analysis can be performed
- Visualization tools like Tableau interpret the values correctly

This is a final step in standardizing the data types for analysis.

In [13]:
# Convert Rating to float
df = df.withColumn(
    "Rating",
    col("Rating").cast("float")
)

## Converting Reviews Column to Integer

The `Reviews` column is converted into an integer data type.

This is important because:
- Reviews represent counts and should be numeric
- Enables aggregation (sum, average, max, etc.)
- Improves compatibility with analysis and visualization tools

This step ensures the column is correctly formatted for analytical operations.

In [16]:
# Convert Reviews to integer
df = df.withColumn(
    "Reviews",
    col("Reviews").cast("int")
)

## Final Schema Check

This step displays the final structure of the DataFrame after all cleaning and type conversions.

It is used to confirm:
- All columns have correct data types
- No unexpected schema changes occurred
- The dataset is ready for analysis and dashboard creation in Tableau

In [17]:
df.printSchema()

root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Reviews: integer (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: integer (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: float (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



## Inspecting Distinct Size Values

This step checks all unique values in the `Size` column.

The purpose is to:
- Understand how app sizes are represented in the dataset
- Identify inconsistencies (e.g., "Varies with device", "k", "M", or null values)
- Support further cleaning and standardization of the column

This helps ensure the `Size` column is reliable for analysis.

In [18]:
# Inspect distinct Size values
df.select("Size").distinct().show(20, truncate=False)

+-----+
|Size |
+-----+
|8.2M |
|970k |
|8.5M |
|691k |
|913k |
|458k |
|245k |
|957k |
|8.6M |
|10.0M|
|317k |
|246k |
|25M  |
|82k  |
|749k |
|193k |
|74M  |
|992k |
|716k |
|176k |
+-----+
only showing top 20 rows


## Handling "Varies with Device" in Size Column

The `Size` column contains values such as "Varies with device", which are not usable for numerical analysis.

In this step:
- These values are replaced with null (`None`)
- This ensures consistency in the dataset
- It prepares the column for proper numeric conversion later

This is an important data cleaning step to standardize missing or undefined values.

In [19]:
# Replace "Varies with device" with null
df = df.withColumn(
    "Size",
    when(col("Size") == "Varies with device", None)
    .otherwise(col("Size"))
)

## Converting Size from KB to MB

Some values in the `Size` column are stored in kilobytes (KB), indicated by the letter "k".

In this step:
- KB values are identified
- The "k" character is removed
- Values are converted into numeric format
- They are converted into megabytes (MB) by dividing by 1024

This ensures all app sizes are standardized into a single unit for analysis.

In [20]:
# Convert KB values to MB
df = df.withColumn(
    "Size",
    when(
        col("Size").contains("k"),
        regexp_replace(col("Size"), "k", "").cast("float") / 1024
    ).otherwise(col("Size"))
)

## Cleaning "M" from Size Column

The `Size` column contains values marked with "M" to represent megabytes.

In this step:
- The "M" character is removed
- The column is standardized into a numeric format

This ensures that all values can be properly converted into a consistent numerical type for analysis.

In [21]:
# Remove "M" from Size column
df = df.withColumn(
    "Size",
    regexp_replace(col("Size"), "M", "")
)

## Converting Size Column to Float

The `Size` column is converted into a float data type.

This step ensures:
- All app sizes are numeric
- Values can be used for statistical analysis
- Compatibility with visualization tools like Tableau

This is the final step in standardizing the `Size` column.

In [22]:
# Convert Size to float
df = df.withColumn(
    "Size",
    col("Size").cast("float")
)

## Final Data Type Standardization

This step performs the final cleaning and standardization of key numeric columns in the dataset.

We ensure that all important analytical fields are correctly formatted:

- **Installs**: cleaned from symbols (+, commas) and converted to integer  
- **Price**: cleaned from `$` and converted to float  
- **Reviews**: converted to integer  
- **Rating**: converted to float  

Finally, the schema is printed to confirm that all transformations were applied successfully and that the dataset is ready for analysis and dashboard creation.

In [33]:
from pyspark.sql import functions as F


# FIX INSTALS

df = df.withColumn(
    "Installs",
    F.regexp_replace(F.col("Installs"), "[+,]", "")
)
df = df.withColumn(
    "Installs",
    F.col("Installs").cast("int")
)


# FIX PRICE

df = df.withColumn(
    "Price",
    F.regexp_replace(F.col("Price"), "[$]", "")
)
df = df.withColumn(
    "Price",
    F.col("Price").cast("float")
)

# FIX REVIEWS
df = df.withColumn(
    "Reviews",
    F.col("Reviews").cast("int")
)


# FIX RATING

df = df.withColumn(
    "Rating",
    F.col("Rating").cast("float")
)


# FINAL VERIFICATION

df.printSchema()

root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Reviews: integer (nullable = true)
 |-- Size: double (nullable = true)
 |-- Installs: integer (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: float (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



## Final Data Cleaning and Validation

This step performs the final data validation and ensures the dataset is fully ready for analysis.

Key actions include:
- Converting all key numeric columns into proper data types (float/int)
- Removing rows with missing or invalid values in important fields
- Performing a final schema check to confirm correctness

This ensures the dataset is clean, consistent, and reliable for dashboard creation in Tableau.

In [ ]:
from pyspark.sql import functions as F


# FORCE SAFE NUMERIC CLEANING


df = df.withColumn("Rating", F.col("Rating").cast("float"))
df = df.withColumn("Reviews", F.col("Reviews").cast("int"))
df = df.withColumn("Installs", F.col("Installs").cast("int"))
df = df.withColumn("Price", F.col("Price").cast("float"))


# REMOVE ROWS WITH INVALID NUMERIC DATA

df = df.dropna(subset=["Rating", "Reviews", "Installs", "Price", "Size"])


# FINAL VALIDATION CHECK


df.printSchema()
df.show(5)

root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Reviews: integer (nullable = true)
 |-- Size: double (nullable = true)
 |-- Installs: integer (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: float (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



+--------------------+--------------+------+-------+----+--------+----+-----+--------------+--------------------+----------------+------------------+------------+
|                 App|      Category|Rating|Reviews|Size|Installs|Type|Price|Content Rating|              Genres|    Last Updated|       Current Ver| Android Ver|
+--------------------+--------------+------+-------+----+--------+----+-----+--------------+--------------------+----------------+------------------+------------+
|Photo Editor & Ca...|ART_AND_DESIGN|   4.1|    159|19.0|   10000|Free|  0.0|      Everyone|        Art & Design| January 7, 2018|             1.0.0|4.0.3 and up|
| Coloring book moana|ART_AND_DESIGN|   3.9|    967|14.0|  500000|Free|  0.0|      Everyone|Art & Design;Pret...|January 15, 2018|             2.0.0|4.0.3 and up|
|U Launcher Lite –...|ART_AND_DESIGN|   4.7|  87510| 8.7| 5000000|Free|  0.0|      Everyone|        Art & Design|  August 1, 2018|             1.2.4|4.0.3 and up|
|Sketch - Draw & P...|

## Creating Final Clean Dataset for Analysis

This step constructs the final cleaned dataset that will be used for analysis and visualization in Tableau.

Key actions include:

- Selecting only relevant columns for analysis
- Ensuring all numeric fields are safely cast to correct data types
- Standardizing key columns such as Rating, Reviews, Installs, Size, and Price
- Removing rows with missing values in critical numeric fields

This produces a clean, structured dataset ready for dashboard development and business insights.

In [ ]:
from pyspark.sql import functions as F


# STEP 1: FORCE CLEAN NUMERIC CASTS USING SAFE CASTING


df_clean = df.select(
    "App",
    "Category",
    F.col("Rating").cast("float").alias("Rating"),
    F.col("Reviews").cast("int").alias("Reviews"),
    F.col("Size").cast("double").alias("Size"),
    F.col("Installs").cast("int").alias("Installs"),
    "Type",
    F.col("Price").cast("float").alias("Price"),
    "Content Rating",
    "Genres",
    "Last Updated",
    "Current Ver",
    "Android Ver"
)


# STEP 2: DROP ROWS WITH NULLS IN KEY NUMERIC FIELD

df_clean = df_clean.dropna(subset=[
    "Rating",
    "Reviews",
    "Installs",
    "Price",
    "Size"
])


# STEP 3: FINAL VALIDATION

df_clean.printSchema()
df_clean.show(5)

root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Reviews: integer (nullable = true)
 |-- Size: double (nullable = true)
 |-- Installs: integer (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: float (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



+--------------------+--------------+------+-------+----+--------+----+-----+--------------+--------------------+----------------+------------------+------------+
|                 App|      Category|Rating|Reviews|Size|Installs|Type|Price|Content Rating|              Genres|    Last Updated|       Current Ver| Android Ver|
+--------------------+--------------+------+-------+----+--------+----+-----+--------------+--------------------+----------------+------------------+------------+
|Photo Editor & Ca...|ART_AND_DESIGN|   4.1|    159|19.0|   10000|Free|  0.0|      Everyone|        Art & Design| January 7, 2018|             1.0.0|4.0.3 and up|
| Coloring book moana|ART_AND_DESIGN|   3.9|    967|14.0|  500000|Free|  0.0|      Everyone|Art & Design;Pret...|January 15, 2018|             2.0.0|4.0.3 and up|
|U Launcher Lite –...|ART_AND_DESIGN|   4.7|  87510| 8.7| 5000000|Free|  0.0|      Everyone|        Art & Design|  August 1, 2018|             1.2.4|4.0.3 and up|
|Sketch - Draw & P...|

## Full Data Cleaning Pipeline (End-to-End)

This is the complete PySpark data cleaning pipeline for the Google Play Store dataset.

### What this pipeline does:

#### 1. Spark Session Initialization
- Creates a Spark session for distributed data processing

#### 2. Data Loading
- Loads the raw CSV dataset into a Spark DataFrame

#### 3. Text Cleaning
- Trims whitespace from key categorical columns such as:
  - App
  - Category
  - Type
  - Content Rating
  - Genres

#### 4. Numeric Cleaning (Safe Casting)
- Rating → converted to double using safe casting
- Reviews → cleaned of non-numeric characters and converted to int
- Installs → cleaned of commas and symbols, converted to int
- Price → removes `$` and converts to double

#### 5. Size Column Cleaning
- Handles mixed formats (KB, MB, “Varies with device”)
- Converts all values into a consistent numeric format (double)

#### 6. Final Dataset Selection
- Selects only relevant columns for analysis
- Ensures correct data types using explicit casting

#### 7. Data Validation
- Removes rows with missing critical values (Rating, Installs)
- Displays final schema and sample rows

#### 8. Data Export
- Saves cleaned dataset as CSV for Tableau dashboard development

This pipeline ensures the dataset is fully cleaned, structured, and ready for business analysis and visualization.

In [1]:
from pyspark.sql import SparkSession, functions as F


# 1. START SPARK SESSION

spark = SparkSession.builder \
    .appName("GooglePlayStoreCleaning") \
    .master("local[*]") \
    .getOrCreate()

# 2. LOAD DATA

df = spark.read.csv(
    "google_play_store_dataset.csv",
    header=True,
    inferSchema=False
)


# 3. CLEAN TEXT COLUMNS SAFELY 

df = df.withColumn("App", F.trim("App")) \
       .withColumn("Category", F.trim("Category")) \
       .withColumn("Type", F.trim("Type")) \
       .withColumn("Content Rating", F.trim("Content Rating")) \
       .withColumn("Genres", F.trim("Genres"))


# 4. CLEAN NUMERIC COLUMNS SAFELY USING try_cast


# Rating
df = df.withColumn(
    "Rating",
    F.expr("try_cast(nullif(Rating, '') as double)")
)

# Reviews (remove non-numeric safely first)
df = df.withColumn(
    "Reviews",
    F.expr("try_cast(regexp_replace(Reviews, '[^0-9]', '') as int)")
)

# Installs
df = df.withColumn(
    "Installs",
    F.expr("try_cast(regexp_replace(Installs, '[^0-9]', '') as int)")
)

# Price (remove $ and convert)
df = df.withColumn(
    "Price",
    F.expr("try_cast(regexp_replace(Price, '[$]', '') as double)")
)


# 5. CLEAN SIZE COLUMN PROPERLY


df = df.withColumn("Size_clean", F.lower(F.trim("Size")))

df = df.withColumn(
    "Size_clean",
    F.when(
        F.col("Size_clean").rlike("varies|unknown|na|nan"),
        None
    ).otherwise(F.col("Size_clean"))
)

size_num = F.regexp_replace(F.col("Size_clean"), "[^0-9.]", "")

df = df.withColumn(
    "Size",
    F.when(
        size_num == "",
        None
    ).when(
        F.col("Size_clean").endswith("k"),
        size_num.cast("double") / 1024
    ).otherwise(
        size_num.cast("double")
    )
)

df = df.drop("Size_clean")

# 6. FINAL SAFE DATASET 

df_clean = df.select(
    "App",
    "Category",
    F.col("Rating").cast("double"),
    F.col("Reviews").cast("int"),
    F.col("Size").cast("double"),
    F.col("Installs").cast("int"),
    "Type",
    F.col("Price").cast("double"),
    "Content Rating",
    "Genres",
    "Last Updated",
    "Current Ver",
    "Android Ver"
)

# OPTIONAL: drop bad rows (recommended for Tableau stability)
df_clean = df_clean.na.drop(subset=["Rating", "Installs"])


# 7. FINAL CHECK

df_clean.printSchema()
df_clean.show(5, truncate=False)


# 8. SAVE FOR TABLEAU 

df_clean.write.mode("overwrite") \
    .option("header", "true") \
    .csv("Google_play_store_cleaned")

print("DONE: Clean dataset successfully saved for Tableau ")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/17 02:40:09 WARN Utils: Your hostname, DESKTOP-UA1KJKB, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/17 02:40:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/17 02:40:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Reviews: integer (nullable = true)
 |-- Size: double (nullable = true)
 |-- Installs: integer (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



+--------------------------------------------------+--------------+------+-------+----+--------+----+-----+--------------+-------------------------+----------------+------------------+------------+
|App                                               |Category      |Rating|Reviews|Size|Installs|Type|Price|Content Rating|Genres                   |Last Updated    |Current Ver       |Android Ver |
+--------------------------------------------------+--------------+------+-------+----+--------+----+-----+--------------+-------------------------+----------------+------------------+------------+
|Photo Editor & Candy Camera & Grid & ScrapBook    |ART_AND_DESIGN|4.1   |159    |19.0|10000   |Free|0.0  |Everyone      |Art & Design             |January 7, 2018 |1.0.0             |4.0.3 and up|
|Coloring book moana                               |ART_AND_DESIGN|3.9   |967    |14.0|500000  |Free|0.0  |Everyone      |Art & Design;Pretend Play|January 15, 2018|2.0.0             |4.0.3 and up|
|U Launche

DONE: Clean dataset successfully saved for Tableau 


26/05/17 18:15:40 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 160034 ms exceeds timeout 120000 ms
26/05/17 18:15:42 WARN SparkContext: Killing executors is not supported by current scheduler.
26/05/17 18:15:48 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

## Data Analysis

### Checking the Final Cleaned Dataset Schema

After completing all data cleaning and transformation steps, we inspect the structure of the final dataset using `printSchema()`.

This step helps us confirm:

* All columns have the correct data types (e.g., `int`, `float`, `double`, `string`)
* Unnecessary or temporary columns have been removed
* The dataset is ready for analysis and visualization in tools like Tableau

It is a final validation step to ensure the cleaned dataset is consistent, well-structured, and suitable for downstream analysis.


In [57]:
df_clean.printSchema()

root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Reviews: integer (nullable = true)
 |-- Size: double (nullable = true)
 |-- Installs: integer (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



### Checking the Number of Records in the Cleaned Dataset

After cleaning and preparing the dataset, we use `df_clean.count()` to determine the total number of rows remaining.

This step helps us:

* Confirm how much data was retained after cleaning
* Understand the impact of removing nulls, duplicates, and invalid values
* Ensure the dataset is still large enough for meaningful analysis
* Validate that no accidental data loss occurred during preprocessing

The final row count gives a quick summary of the dataset size before moving into analysis and visualization stages.


In [58]:
df_clean.count()

9364

###  Summary Statistics of the Cleaned Dataset

The `df_clean.describe().show()` command is used to generate basic statistical summaries for all numeric columns in the dataset.

This step helps us understand the overall distribution and characteristics of the data after cleaning.

It provides key metrics such as:

* **Count** → number of non-null values
* **Mean** → average value
* **Standard deviation** → spread of values
* **Minimum and maximum values** → range of the data

This is an important part of Exploratory Data Analysis (EDA) because it helps identify:

* Outliers
* Skewed distributions
* Potential anomalies in the cleaned dataset

The results confirm whether the data is now stable and reliable for further analysis and visualization.


In [59]:
df_clean.describe().show()

26/05/16 13:11:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+--------------+------------------+------------------+------------------+-------------------+----+------------------+---------------+------+-----------------+-------------+------------------+
|summary|                 App|      Category|            Rating|           Reviews|              Size|           Installs|Type|             Price| Content Rating|Genres|     Last Updated|  Current Ver|       Android Ver|
+-------+--------------------+--------------+------------------+------------------+------------------+-------------------+----+------------------+---------------+------+-----------------+-------------+------------------+
|  count|                9364|          9364|              9364|              9364|              7728|               9364|9364|              9364|           9364|  9364|             9364|         9363|              9364|
|   mean|                NULL|          NULL| 4.191734301580527| 514148.4421187527|22.959594410645476|1.790062558073

###  Checking Missing Values in the Cleaned Dataset

This step is used to verify the number of null (missing) values in each column of the cleaned dataset.

We use PySpark functions to:

* Loop through every column in `df_clean`
* Count how many rows contain `NULL` values per column
* Display the results in a structured format

This is important because:

* It confirms whether data cleaning was successful
* It helps identify any remaining missing values that could affect analysis
* It ensures the dataset is ready for visualization tools like Tableau without unexpected gaps

A clean dataset should ideally have **very few or no null values in key analytical columns** such as:

* Rating
* Installs
* Reviews
* Price

This step acts as a final data quality check before moving to deeper analysis and dashboard creation.


In [60]:
from pyspark.sql import functions as F

df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_clean.columns
]).show()

+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+
|App|Category|Rating|Reviews|Size|Installs|Type|Price|Content Rating|Genres|Last Updated|Current Ver|Android Ver|
+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+
|  0|       0|     0|      0|1636|       0|   0|    0|             0|     0|           0|          1|          0|
+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+



###  App Distribution by Category (Top 10 Categories)

This analysis groups the dataset by **app category** and counts how many apps exist in each category.

We then sort the results in descending order to identify the most populated categories in the Google Play Store dataset.

#### What this tells us:

* Which categories have the **highest number of apps**
* How competitive or saturated each category is
* Where most developers are focusing their efforts

#### Why this is useful:

* Helps the startup understand **market saturation**
* Supports decision-making on **which category to target**
* Provides insight into **industry trends and demand areas**

By examining the top categories, we can better understand where opportunities or competition are strongest in the Android app ecosystem.


In [61]:
df_clean.groupBy("Category").count().orderBy("count", ascending=False).show(10)

+---------------+-----+
|       Category|count|
+---------------+-----+
|         FAMILY| 1747|
|           GAME| 1097|
|          TOOLS|  734|
|   PRODUCTIVITY|  351|
|        MEDICAL|  350|
|  COMMUNICATION|  328|
|        FINANCE|  323|
|         SPORTS|  319|
|    PHOTOGRAPHY|  317|
|PERSONALIZATION|  314|
+---------------+-----+
only showing top 10 rows


###  Distribution of App Types (Free vs Paid)

This step groups the dataset by the **Type** column and counts how many apps fall into each category (e.g., Free or Paid).

#### What this analysis shows:

* The proportion of **Free apps vs Paid apps** in the Google Play Store dataset
* The dominant pricing model used by developers
* Basic insight into monetization strategies in the market

#### Why it matters:

* Helps understand whether the market is mostly **free-driven or paid-driven**
* Supports business decisions around **pricing strategy**
* Gives context for further analysis on revenue potential and user adoption

This is a key early indicator of how apps are typically positioned in the marketplace.


In [62]:
df_clean.groupBy("Type").count().show()

+----+-----+
|Type|count|
+----+-----+
|Free| 8717|
|Paid|  647|
+----+-----+



###  Summary Statistics for App Ratings

This step generates a statistical summary specifically for the **Rating** column using `summary()`.

Unlike `describe()`, which focuses on general numeric summaries, `summary()` provides more detailed percentiles and distribution insights.

#### What this output includes:

* **Count** → number of non-null ratings
* **Mean** → average app rating
* **Standard deviation** → variability in ratings
* **Min / Max** → lowest and highest ratings
* **Percentiles (25%, 50%, 75%)** → distribution breakdown

#### Why this is useful:

* Helps understand how users generally rate apps
* Identifies whether ratings are clustered (e.g., most apps highly rated)
* Detects skewness or unusual rating patterns
* Supports later analysis on **what makes apps successful**

This is an important step for understanding overall **user satisfaction trends** in the dataset.


In [63]:
df_clean.select("Rating").summary().show()

+-------+------------------+
|summary|            Rating|
+-------+------------------+
|  count|              9364|
|   mean| 4.191734301580527|
| stddev|0.5152693809765408|
|    min|               1.0|
|    25%|               4.0|
|    50%|               4.3|
|    75%|               4.5|
|    max|               5.0|
+-------+------------------+



###  Correlation Between Rating and Installs

This step calculates the **correlation** between `Rating` and `Installs` using:

```python
df_clean.select("Rating", "Installs", "Reviews", "Price", "Size").corr("Rating", "Installs")
```

#### What this means:

* **Correlation measures the relationship** between two numeric variables
* It tells us whether higher-rated apps tend to have more installs

#### How to interpret the result:

* **+1 → strong positive relationship** (as rating increases, installs increase)
* **0 → no relationship**
* **-1 → negative relationship** (as rating increases, installs decrease)

#### Why this is important:

* Helps us understand if **quality (rating)** is linked to **popularity (installs)**
* Supports business decisions like:

  * Should we focus on improving app quality?
  * Or is visibility/marketing more important than ratings?

This is a key insight step in understanding what drives app success in the Google Play Store ecosystem.


In [64]:
df_clean.select("Rating", "Installs", "Reviews", "Price", "Size").corr("Rating", "Installs")

0.05136158255759884

###  Exploratory Data Analysis (EDA) – Business Insights

This section performs **initial analytical exploration** of the cleaned Google Play Store dataset to extract meaningful business insights.

It focuses on understanding:

* Dataset structure
* Category performance
* User engagement signals
* Pricing behavior

---

##  1. Dataset Overview

We first check the total number of records and schema structure.

### Why this matters:

* Confirms dataset size after cleaning
* Ensures correct data types for analysis

---

##  2. Top App Categories

We group apps by **Category** and count how many apps exist in each.

### Insight:

* Shows which categories dominate the Play Store
* Helps identify saturated vs emerging markets

---

##  3. Average Rating by Category

We compute the **average rating per category**.

### Insight:

* Reveals which categories have the most satisfied users
* Helps identify high-quality app segments

---

##  4. Top Apps by Reviews

We rank apps based on number of **Reviews**.

### Insight:

* Identifies most engaged/popular apps
* Reviews often indicate **user activity and trust**

---

##  5. Free vs Paid Apps Analysis

We create a new variable (`App_Type`) based on Price:

* Free = Price == 0
* Paid = Price > 0

Then we count distribution.

### Insight:

* Shows dominant monetization model
* Helps understand market pricing strategy

---

##  Overall Value of This Section

This EDA phase helps answer key business questions:

* What types of apps dominate the market?
* Which categories perform best in ratings?
* Are users more engaged with certain apps?
* Is the market mainly free or paid?

These insights form the foundation for the **dashboard design in Tableau**.

---




In [56]:
# Analysis Code Here - if needed
from pyspark.sql import functions as F


# 1. BASIC OVERVIEW

print("TOTAL ROWS:")
print(df_clean.count())

print("\nSCHEMA:")
df_clean.printSchema()


# 2. TOP CATEGORIES BY NUMBER OF APPS
print("\nTOP APP CATEGORIES:")
df_clean.groupBy("Category") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10, truncate=False)


# 3. AVERAGE RATING BY CATEGORY

print("\nAVERAGE RATING BY CATEGORY:")
df_clean.groupBy("Category") \
    .agg(F.avg("Rating").alias("avg_rating")) \
    .orderBy(F.desc("avg_rating")) \
    .show(10, truncate=False)


# 4. TOP APPS BY REVIEWS
print("\nTOP APPS BY REVIEWS:")
df_clean.orderBy(F.desc("Reviews")) \
    .select("App", "Reviews", "Rating") \
    .show(10, truncate=False)


# 5. PRICE INSIGHT (FREE VS PAID)

print("\nFREE VS PAID APPS:")
df_clean.withColumn(
    "App_Type",
    F.when(F.col("Price") == 0, "Free").otherwise("Paid")
).groupBy("App_Type") \
 .count() \
 .show()

TOTAL ROWS:


9364

SCHEMA:
root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Reviews: integer (nullable = true)
 |-- Size: double (nullable = true)
 |-- Installs: integer (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)


TOP APP CATEGORIES:


+---------------+-----+
|Category       |count|
+---------------+-----+
|FAMILY         |1747 |
|GAME           |1097 |
|TOOLS          |734  |
|PRODUCTIVITY   |351  |
|MEDICAL        |350  |
|COMMUNICATION  |328  |
|FINANCE        |323  |
|SPORTS         |319  |
|PHOTOGRAPHY    |317  |
|PERSONALIZATION|314  |
+---------------+-----+
only showing top 10 rows

AVERAGE RATING BY CATEGORY:


+-------------------+------------------+
|Category           |avg_rating        |
+-------------------+------------------+
|EVENTS             |4.435555555555557 |
|EDUCATION          |4.389032258064517 |
|ART_AND_DESIGN     |4.358064516129031 |
|BOOKS_AND_REFERENCE|4.346067415730338 |
|PERSONALIZATION    |4.335987261146501 |
|PARENTING          |4.300000000000001 |
|GAME               |4.2863263445761195|
|BEAUTY             |4.278571428571428 |
|HEALTH_AND_FITNESS |4.2773648648648654|
|SHOPPING           |4.259663865546221 |
+-------------------+------------------+
only showing top 10 rows

TOP APPS BY REVIEWS:


+----------------------------------------+--------+------+
|App                                     |Reviews |Rating|
+----------------------------------------+--------+------+
|Facebook                                |78158306|4.1   |
|Facebook                                |78128208|4.1   |
|WhatsApp Messenger                      |69119316|4.4   |
|WhatsApp Messenger                      |69119316|4.4   |
|WhatsApp Messenger                      |69109672|4.4   |
|Instagram                               |66577446|4.5   |
|Instagram                               |66577313|4.5   |
|Instagram                               |66577313|4.5   |
|Instagram                               |66509917|4.5   |
|Messenger – Text and Video Chat for Free|56646578|4.0   |
+----------------------------------------+--------+------+
only showing top 10 rows

FREE VS PAID APPS:


+--------+-----+
|App_Type|count|
+--------+-----+
|    Free| 8717|
|    Paid|  647|
+--------+-----+



### Link to Published Dashboard

https://public.tableau.com/views/GooglePlayStore_17789542437770/FinalInsightsDashboard?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link

## Conclusion

Markdown here

 Project Documentation (Google Play Store Analysis)

##  1. Project Overview

This project analyzes the Google Play Store dataset to understand trends in app success, user engagement, and monetization strategies.

The goal is to support a startup planning to enter the Android app market by identifying:

* High-performing app categories
* Relationship between ratings, installs, and reviews
* Pricing strategies (free vs paid apps)
* Key factors influencing app success

The analysis follows the CRISP-DM methodology:

* Business Understanding
* Data Understanding
* Data Preparation
* Data Analysis
* Visualization (Tableau – upcoming step)

---

##  2. Data Cleaning Summary

The raw dataset contained inconsistencies such as:

* Non-numeric values in numeric columns
* Missing and malformed entries
* Special characters in fields like Installs and Price
* Inconsistent formatting in Size column
* Duplicate rows

### Key cleaning steps performed:

#### ✔ Text Standardization

* Trimmed whitespace from columns like App, Category, Type, Genres

#### ✔ Numeric Conversions

* Converted Rating → float
* Cleaned Reviews → integer
* Cleaned Installs → removed “+” and “,” then converted to integer
* Cleaned Price → removed “$” and converted to float

#### ✔ Size Column Processing

* Handled values like “Varies with device”
* Converted KB values to MB
* Removed text characters and standardized numeric format

#### ✔ Missing Value Handling

* Removed rows with null values in key analytical columns:

  * Rating
  * Installs
  * Reviews
  * Price
  * Size

#### ✔ Duplicate Removal

* Removed duplicate records to ensure data integrity

---

##  3. Data Challenges & Issues

Several issues were encountered in the dataset:

* Mixed data formats in numeric columns (e.g., "1,000,000+", "$4.99")
* Missing values in important fields
* Inconsistent size formats (M, k, “Varies with device”)
* Spark casting errors due to empty strings
* Need for safe parsing using `try_cast` and regex cleaning

These issues required careful preprocessing to avoid analysis errors.

---

##  4. Assumptions Made

* Missing values in key numeric columns were removed rather than imputed
* “Varies with device” was treated as missing data in Size column
* Price = 0 was assumed to represent free apps
* Duplicate rows were considered invalid and removed
* Ratings outside valid numeric range were excluded implicitly during cleaning

---

##  5. Final Dataset Structure

After cleaning, the dataset contains:

* App (string)
* Category (string)
* Rating (float)
* Reviews (int)
* Size (float, MB)
* Installs (int)
* Type (Free/Paid)
* Price (float)
* Content Rating (string)
* Genres (string)
* Last Updated (string)
* Current Version (string)
* Android Version (string)

---

##  6. Output Dataset

The cleaned dataset was exported as a CSV file:

> `Google_play_store_cleaned`

This file is ready for:

* Tableau dashboard creation
* Business intelligence reporting
* Further statistical analysis

---

##  7. Key Takeaways from Cleaning Phase

* Data required significant preprocessing before analysis
* Numeric conversion issues were the most common challenge
* Proper use of `regexp_replace` and `try_cast` ensured data integrity
* Final dataset is now consistent, structured, and analysis-ready

---

##  8. Next Step

The cleaned dataset will be used to:

* Build Tableau visualizations (minimum 6 required)
* Create an interactive dashboard
* Answer key business questions about app success factors

---





#  Step 7.1: AI Integration & Usage Documentation

##  Use of Generative AI in this Project

Generative AI (ChatGPT) was used as a supporting tool throughout this data analysis project to accelerate development, improve code quality, and assist with debugging.

The AI was used as an **assistant**, not a replacement for analytical thinking. All outputs were reviewed, tested, and adjusted manually where necessary.

---

##  1. Areas Where AI Was Used

### ✔ Code Development Support

AI was used to:

* Generate PySpark data cleaning pipelines
* Assist with safe type conversions using `try_cast` and `regexp_replace`
* Help structure EDA queries (groupBy, aggregations, correlations)
* Improve readability and organization of code blocks

---

### ✔ Debugging & Error Resolution

AI helped resolve:

* Spark `CAST_INVALID_INPUT` errors
* Empty string conversion issues in numeric fields
* Column formatting issues in Size, Price, and Installs columns
* Data type mismatch problems during transformations

---

### ✔ Data Cleaning Optimization

AI suggested improvements such as:

* Using `try_cast` for safer numeric conversions
* Standardizing cleaning logic across multiple columns
* Reducing repetitive transformations
* Improving pipeline structure for readability and efficiency

---

### ✔ Documentation Support

AI was used to:

* Structure CRISP-DM documentation sections
* Write explanations for EDA and cleaning steps
* Ensure clarity and academic presentation quality

---

##  2. Human Oversight & Validation

All AI-generated outputs were:

* Tested in PySpark environment
* Debugged for runtime errors
* Adjusted for dataset-specific issues
* Verified against expected outputs

Final decisions (such as column dropping, null handling, and filtering rules) were made by the analyst.

---

##  3. Key Principle Followed

> “AI was used as an assistant for speed and structure, while all analytical decisions and validations were handled by the human analyst.”

---

##  4. Impact of AI on Workflow

Using AI significantly:

* Reduced development time for PySpark pipelines
* Helped debug complex Spark errors faster
* Improved code organization and readability
* Allowed focus on insights rather than boilerplate code

---

##  5. Summary

AI played a supportive role in:

* Data cleaning
* Code optimization
* Error debugging
* Documentation structuring

However, the final dataset, analysis decisions, and project direction were fully controlled and validated by the analyst.

